In [1]:
# extract the Paris data from the csv files with elections for all municipalities of France
# code written with some Gemini tinkering

import pandas as pd

input_file = 'municipales26_tour1.csv'
output_file = 'municipales26_tour1_paris.csv'

df = pd.read_csv(input_file, sep=';', on_bad_lines='warn', low_memory=False)


# Keywords to exclude
forbidden_prefixes = [
    "Libellé de liste", "Nom candidat", "Prénom candidat", 
    "Numéro de panneau", "Sièges", "Elu", "Sexe", "Libellé abrégé", "% Voix/inscrits"
]

forbidden_suffixes = ["10", "11", "12", "13"]

# Identify columns to keep: 
# Must be index >= 4  and not start with forbidden prefixes
cols_to_keep = []
for i, col in enumerate(df.columns):
    if 4 <= i:
        if not any(col.startswith(prefix) for prefix in forbidden_prefixes) and not any(col.endswith(suffix) for suffix in forbidden_suffixes):
            cols_to_keep.append(col)

# Note: We still use df.iloc on the original dataframe to find our filter values
# because the first 4 columns (indices 0 and 2) are needed for the logic.
c1_vals = pd.to_numeric(df.iloc[:, 0], errors='coerce')

mask = (c1_vals == 75) 

final_df = df.loc[mask, cols_to_keep]

# save the extracted data to the output file
final_df.to_csv(output_file, index=False, sep=';', encoding='utf-8')

print(f"Success! {len(final_df)} rows and {len(cols_to_keep)} columns saved.")

Success! 903 rows and 42 columns saved.


In [2]:
# convert the percent strings into number. e.g. "2,35%" becomes 2.35

file_name = 'municipales26_tour1_paris.csv'

# 1. Load the file
df = pd.read_csv(file_name, sep=';', low_memory=False)

# 2. Iterate through columns and clean percentages
for col in df.columns:
    # Check if the column is of type 'object' (string) 
    # and if it contains the '%' character in the first non-null row
    sample_value = df[col].dropna().astype(str).iloc[0] if not df[col].dropna().empty else ""
    
    if "%" in sample_value:
        # Remove %, replace comma with dot (common in French data), and convert to float
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('%', '', regex=False)
            .str.replace(',', '.', regex=False) # Handles "2,30%" format
            .astype(float)
        )

# 3. Save the cleaned file
df.to_csv(file_name, index=False, sep=';', encoding='utf-8')

print("Percentage columns have been converted to numeric values.")

Percentage columns have been converted to numeric values.


In [3]:
# simplify by summing the 3 far-left parties together

file_name = 'municipales26_tour1_paris.csv'

# Using sep=';' as established in previous steps
df = pd.read_csv(file_name, sep=';', low_memory=False)


# We use .strip() to handle potential hidden spaces
lexg_indices = []
for i in range(len(df.columns)):
    # Check if the column is entirely "LEXG" (ignoring NaNs)
    column_values = df.iloc[:, i].astype(str).str.strip()
    if (column_values == "LEXG").all():
        lexg_indices.append(i)

# These columns contain the vote counts
voix_columns_data = []
for idx in lexg_indices:
    if idx + 1 < len(df.columns):
        # Convert to numeric, turning errors to 0 so the sum doesn't fail
        voix_col = pd.to_numeric(df.iloc[:, idx + 1], errors='coerce').fillna(0)
        voix_columns_data.append(voix_col)

voix_perc_columns_data = []
for idx in lexg_indices:
    if idx + 1 < len(df.columns):
        # Convert to numeric, turning errors to 0 so the sum doesn't fail
        voix_col = pd.to_numeric(df.iloc[:, idx + 2], errors='coerce').fillna(0)
        voix_perc_columns_data.append(voix_col)

if voix_columns_data:
    # sum(list_of_series) adds them row-by-row
    df['Voix TOT_LEXG'] = sum(voix_columns_data)

if voix_columns_data:
    # sum(list_of_series) adds them row-by-row
    df['% Voix/Exprimés TOT ExG'] = sum(voix_perc_columns_data)


df.to_csv(file_name, index=False, sep=';', encoding='utf-8')


In [4]:
# finally, eliminate the columns corresponding to the individual scores of the 3 far-left parties

file_name = 'municipales26_tour1_paris.csv'

# 1. Load the current file
df = pd.read_csv(file_name, sep=';', low_memory=False, encoding='utf-8')
# 2. Identify indices of columns to drop
indices_to_drop = []
cols = df.columns

for i in range(len(cols)):
    # Check if the column is a "LEXG" marker column
    column_values = df[cols[i]].astype(str).str.strip()
    if (column_values == "LEXG").all():
        # Add the LEXG column index
        indices_to_drop.append(i)
        # Add the next two column indices (e.g., Voix and %)
        if i + 1 < len(cols):
            indices_to_drop.append(i + 1)
        if i + 2 < len(cols):
            indices_to_drop.append(i + 2)

# 3. Drop the columns by index
# We use unique() to avoid errors if "LEXG" columns are close together
# and convert to a list of names to drop
columns_to_remove = [cols[i] for i in sorted(set(indices_to_drop))]
df_cleaned = df.drop(columns=columns_to_remove)
df_cleaned.columns = [col.replace('é', 'e') for col in df_cleaned.columns]
df_cleaned.columns = [col.replace('%', 'perc') for col in df_cleaned.columns]


# 4. Save the final version
df_cleaned.to_csv(file_name, index=False, sep=';', encoding='utf-8')

print(f"Cleanup complete. Removed {len(columns_to_remove)} columns.")
print(f"Remaining columns: {len(df_cleaned.columns)}")

Cleanup complete. Removed 9 columns.
Remaining columns: 35


In [5]:
# now, the data for the second round

input_file = 'municipales26_tour2.csv'
output_file = 'municipales26_tour2_paris.csv'

df = pd.read_csv(input_file, sep=';', on_bad_lines='warn', low_memory=False)


# Keywords to exclude
forbidden_prefixes = [
    "Libellé de liste", "Nom candidat", "Prénom candidat", 
    "Numéro de panneau", "Sièges", "Elu", "Sexe", "Libellé abrégé", "% Voix/inscrits"
]

forbidden_suffixes = ["4", "5"]

# Identify columns to keep: 
# Must be index >= 4  and not start with forbidden prefixes
cols_to_keep = []
for i, col in enumerate(df.columns):
    if 4 <= i:
        if not any(col.startswith(prefix) for prefix in forbidden_prefixes) and not any(col.endswith(suffix) for suffix in forbidden_suffixes):
            cols_to_keep.append(col)

# Note: We still use df.iloc on the original dataframe to find our filter values
# because the first 4 columns (indices 0 and 2) are needed for the logic.
c1_vals = pd.to_numeric(df.iloc[:, 0], errors='coerce')

mask = (c1_vals == 75) 

final_df = df.loc[mask, cols_to_keep]

for col in final_df.columns:
    # Check if the column is of type 'object' (string) 
    # and if it contains the '%' character in the first non-null row
    sample_value = final_df[col].dropna().astype(str).iloc[0] if not final_df[col].dropna().empty else ""
    
    if "%" in sample_value:
        # Remove %, replace comma with dot (common in French data), and convert to float
        final_df[col] = (
            df[col]
            .astype(str)
            .str.replace('%', '', regex=False)
            .str.replace(',', '.', regex=False) # Handles "2,30%" format
            .astype(float)
        )


final_df.columns = [col.replace('é', 'e') for col in final_df.columns]
final_df.columns = [col.replace('%', 'perc') for col in final_df.columns]


final_df.to_csv(output_file, index=False, sep=';', encoding='utf-8')

print(f"Success! {len(final_df)} rows and {len(cols_to_keep)} columns saved.")

Success! 903 rows and 24 columns saved.
